# ReconAnything keep-alive (Colab / Kaggle)

Keeps the Render free-tier services awake the same way Forge 3D stays up: an always-on external poller that hits the services on a short interval, so they never sleep even when both are idle at once.

1. Edit `URLS` below if your service names differ.
2. **Runtime -> Run all** (Colab) or **Run All** (Kaggle).
3. Leave the tab open. Re-run when the host ends the session (Colab ~12h, Kaggle ~9h). `MAX_HOURS` makes it exit cleanly just before that.

Stdlib only -- no `pip install`.

In [ ]:
import random, time, urllib.error, urllib.request
from datetime import datetime, timezone

URLS = [
    "https://recon-anything.onrender.com/api/health",
    "https://recon-anything-worker.onrender.com/",
]
INTERVAL_S = 240          # ~4 min, well under Render's ~15 min sleep window
MAX_HOURS  = 8            # exit cleanly before the host kills the session; 0 = forever
COLD_START_TIMEOUT = 90   # a sleeping instance can take ~60s to wake

def now():
    return datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%SZ')

def ping(url):
    t0 = time.time()
    req = urllib.request.Request(url, method='GET',
                                 headers={'User-Agent': 'recon-keepalive-poller/1'})
    try:
        with urllib.request.urlopen(req, timeout=COLD_START_TIMEOUT) as r:
            return (r.status < 500, r.status, time.time() - t0)
    except urllib.error.HTTPError as e:
        return (e.code < 500, e.code, time.time() - t0)
    except Exception:
        return (False, 0, time.time() - t0)

print(f'[{now()}] keepalive started: {len(URLS)} url(s), every ~{INTERVAL_S}s, '
      + (f'stopping after {MAX_HOURS}h' if MAX_HOURS else 'forever'))
started = time.time()
fails = {u: 0 for u in URLS}
while True:
    for u in URLS:
        ok, status, dt = ping(u)
        fails[u] = 0 if ok else fails[u] + 1
        flag = 'ok  ' if ok else 'FAIL'
        warn = f'  (x{fails[u]} in a row)' if fails[u] else ''
        print(f'[{now()}] {flag} {status:>3} {dt:5.1f}s  {u}{warn}', flush=True)
    if MAX_HOURS and (time.time() - started) >= MAX_HOURS * 3600:
        print(f'[{now()}] reached max hours, exiting cleanly'); break
    time.sleep(INTERVAL_S + random.uniform(-15, 15))